# Phase 4: Daily Dataset Creation
## Daily Electricity Consumption Prediction in Smart Buildings (BDG2 Dataset)

- **Project Title:** Daily Electricity Consumption Prediction in Smart Buildings Using Machine Learning
- **Dataset:** Building Data Genome Project 2 (BDG2)
- **Academic Context:** 4-Member Team Academic ML Project (3rd Year ISE)
- **Phase Objective:** Aggregate cleaned hourly energy readings and meteorological data into the primary processed daily modeling dataset:
  `data/processed/daily_energy_dataset.csv`
- **Target Variable:** Daily electricity consumption in **kilowatt-hours per day (kWh/day)**


### 1. Mathematical Aggregation Framework & Simple Examples

#### Energy vs. Power Verification:
- **Meter Type:** Electricity meter.
- **Physical Unit:** Kilowatt-hours (kWh) per 1-hour interval. This is **cumulative electrical energy**, not instantaneous power in kilowatts (kW).
- **Time Frequency:** Exactly 1-hour intervals (`00:00:00`, `01:00:00`, ..., `23:00:00`).

#### Concrete Aggregation Examples:

1. **Standard 24-Hour Day (Case 1):**
   A building has 24 valid hourly energy measurements $E_1, E_2, \dots, E_{24}$:
   $$\text{Daily Energy (kWh/day)} = \sum_{h=1}^{24} E_h$$
   *Example:* If hourly readings average 10.0 kWh each hour, $\text{Daily Energy} = 24 \times 10.0 = 240.0\text{ kWh/day}$.

2. **Daylight Saving Time (DST) Spring-Forward Day (Case 2 - 23 Hours):**
   On the second Sunday in March, local time skips 02:00 to 03:00, resulting in 23 physical hours.
   Valid hours $N = 23$. Raw sum = $230.0\text{ kWh}$.
   To prevent under-representing daily energy consumption relative to standard 24-hour days:
   $$\text{Daily Energy}_{\text{normalized}} = \left(\sum_{h=1}^{23} E_h\right) \times \frac{24}{23} = 230.0 \times \frac{24}{23} = 240.0\text{ kWh/day}$$

3. **Incomplete Day with Hardware Failure / Outage (Case 3 - e.g., 14 Hours):**
   Suppose only 14 hourly readings were recorded ($140\text{ kWh}$) because the data logger disconnected.
   If blindly summed, this day would be recorded as $140\text{ kWh}$—an artificial 42% drop.
   Because valid hours ($14$) is below our strict quality cutoff ($N \ge 23$), this day is classified as **Incomplete** and discarded.


### 2. Environment Setup & Project Imports
We import standard data science tools and our modular daily aggregator utility from `src/daily_aggregator.py`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 10

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.daily_aggregator import aggregate_daily_weather, aggregate_daily_electricity, create_daily_dataset

processed_dir = project_root / "data" / "processed"
interim_dir = project_root / "data" / "interim"
tables_dir = project_root / "results" / "tables"

print(f"Project Root: {project_root}")
print(f"Processed Directory: {processed_dir}")


### 3. Inspecting the Processed Daily Energy Dataset
Let us load the generated `data/processed/daily_energy_dataset.csv` and inspect its shape, column schema, and data types.


In [ ]:
csv_path = processed_dir / "daily_energy_dataset.csv"
daily_df = pd.read_csv(csv_path)

print(f"Daily Dataset Dimensions: {daily_df.shape[0]:,} rows x {daily_df.shape[1]} columns")
print(f"Unique Buildings: {daily_df['building_id'].nunique()}")
print(f"Date Range: {daily_df['date'].min()} to {daily_df['date'].max()} ({daily_df['date'].nunique()} unique calendar dates)")
display(daily_df.head(5))


### 4. Schema and Data Quality Verification
We check data types, column descriptions, and verify that there are zero nulls in critical target or weather features.


In [ ]:
print("--- Column Data Types & Missing Value Audit ---")
schema_df = pd.DataFrame({
    "Data Type": daily_df.dtypes,
    "Non-Null Count": daily_df.notnull().sum(),
    "Missing Count": daily_df.isnull().sum(),
    "Missing %": (daily_df.isnull().mean() * 100).round(2)
})
display(schema_df)


### 5. Daily Electricity Target Distribution
Let us visualize the distribution of daily electricity consumption across the building portfolio in kilowatt-hours per day ($	ext{kWh/day}$).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Distribution of Daily Consumption for a Sample Building
sample_bldg = "Panther_education_Tina"
sample_df = daily_df[daily_df["building_id"] == sample_bldg]

sns.histplot(sample_df["meter_reading_daily"], bins=40, kde=True, color="#2980b9", ax=axes[0], edgecolor="black")
axes[0].set_title(f"Daily Electricity Consumption ({sample_bldg})")
axes[0].set_xlabel("Daily Consumption (kWh / day)")
axes[0].set_ylabel("Number of Days")
axes[0].axvline(sample_df["meter_reading_daily"].mean(), color="red", linestyle="--", label=f"Mean: {sample_df['meter_reading_daily'].mean():.1f} kWh/day")
axes[0].axvline(sample_df["meter_reading_daily"].median(), color="gold", linestyle="-", label=f"Median: {sample_df['meter_reading_daily'].median():.1f} kWh/day")
axes[0].legend()

# Subplot 2: Breakdown of Full 24-Hour vs Normalized 23-Hour Days
day_type_counts = daily_df["valid_hours"].value_counts()
axes[1].bar(["Full Days (24 Hours)", "Normalized Days (23 Hours)"], day_type_counts.values, color=["#27ae60", "#f39c12"], edgecolor="black")
axes[1].set_title("Distribution of Full vs. Normalized Days")
axes[1].set_ylabel("Total Building-Days")
for i, v in enumerate(day_type_counts.values):
    pct = v / len(daily_df) * 100
    axes[1].text(i, v + 15000, f"{v:,}\n({pct:.2f}%)", ha="center", fontweight="bold")
axes[1].set_ylim(0, 1150000)

plt.tight_layout()
plt.show()


### 6. Aggregation Summary Audit Table
Let us display the official audit summary table saved in `results/tables/03_daily_aggregation_summary.csv`.


In [ ]:
summary_table = pd.read_csv(tables_dir / "03_daily_aggregation_summary.csv")
display(summary_table)


### 7. Phase 4 Conclusion & Hand-Off to Phase 5
- **Verified Target:** Daily electricity consumption is strictly defined in **kilowatt-hours per day (kWh/day)**.
- **Robust Incomplete Day Filtering:** Discarded 47,748 building-days with $<23$ valid hours (4.54% of possible days) to guarantee that hardware telemetry dropouts do not pollute model training.
- **Dataset Availability:** `data/processed/daily_energy_dataset.csv` contains **1,004,892 verified observations** ready for feature engineering in Phase 5.
